In [ ]:
# @title Environment Setup & Package Installation
!pip install -q google-adk litellm "google-cloud-aiplatform[adk,agent_engines]" requests

In [ ]:
# @title Google Cloud Project & API Key Configuration
import os
import getpass
import google.auth
import vertexai

try:
    _, PROJECT_ID = google.auth.default()
except Exception:
    PROJECT_ID = None

if not PROJECT_ID:
    PROJECT_ID = input("Enter your lab Project ID: ").strip()

LOCATION = "us-central1"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"

MAPS_API_KEY = getpass.getpass("Google Maps Geocoding API key: ")
os.environ["MAPS_API_KEY"] = MAPS_API_KEY

MODEL_GEMINI = "gemini-2.5-flash"
vertexai.init(project=PROJECT_ID, location=LOCATION)

print(f"Project: {PROJECT_ID}")
print(f"Maps key loaded: {bool(MAPS_API_KEY)}")

Google Maps Geocoding API key: ··········
Project: qwiklabs-gcp-04-79b727f6c308
Maps key loaded: True


In [ ]:
# @title Tool Definition: Geocoding Function (get_lat_lon)
import os
from typing import Dict, Optional
import requests

def get_lat_lon(place: str) -> Optional[Dict[str, float]]:
    """Convert a place name into latitude and longitude using the Google Maps Geocoding API."""
    try:
        response = requests.get(
            "https://maps.googleapis.com/maps/api/geocode/json",
            params={"address": place, "key": os.environ["MAPS_API_KEY"]},
            timeout=10,
        )
        response.raise_for_status()
        data = response.json()
        if data.get("status") != "OK" or not data.get("results"):
            return None
        location = data["results"][0]["geometry"]["location"]
        return {"lat": location["lat"], "lon": location["lng"]}
    except (requests.RequestException, KeyError, ValueError):
        return None

print(get_lat_lon("Denver, CO"))

{'lat': 39.7392358, 'lon': -104.990251}


In [ ]:
# @title Tool Definition: Weather Forecast Function (get_extended_weather_forecast)
from typing import Dict, List, Optional
import requests

NWS_HEADERS = {"User-Agent": "(adk-skills-workshop, your-email@example.com)"}

def get_extended_weather_forecast(lat: float, lon: float) -> Optional[List[Dict[str, str]]]:
    """Fetch the extended weather forecast from the U.S. National Weather Service API."""
    try:
        points_response = requests.get(
            f"https://api.weather.gov/points/{lat},{lon}",
            headers=NWS_HEADERS,
            timeout=10,
        )
        points_response.raise_for_status()
        forecast_url = points_response.json()["properties"]["forecast"]

        forecast_response = requests.get(forecast_url, headers=NWS_HEADERS, timeout=10)
        forecast_response.raise_for_status()

        periods = forecast_response.json()["properties"]["periods"]
        return [
            {
                "name": p["name"],
                "temperature": f"{p['temperature']} {p['temperatureUnit']}",
                "wind": f"{p['windSpeed']} {p['windDirection']}",
                "short_forecast": p["shortForecast"],
                "detailed_forecast": p["detailedForecast"]
            }
            for p in periods[:6]
        ]
    except (requests.RequestException, KeyError, ValueError):
        return None

print("Weather tool ready.")

Weather tool ready.


In [ ]:
# @title Agent Instructions & Callbacks Configuration
import logging
from logging.handlers import RotatingFileHandler
from typing import Optional
from google.adk.agents import Agent
from google.adk.agents.callback_context import CallbackContext
from google.adk.models import LlmRequest, LlmResponse

WEATHER_AGENT_INSTRUCTIONS = """
You are Pat, a friendly U.S. weather assistant.

Workflow:
1. When a user names a location, call get_lat_lon to resolve it to coordinates.
2. Pass those coordinates to get_extended_weather_forecast.
3. Summarize the weather in 3-5 sentences of plain language.
4. If conditions are hazardous - severe storms, extreme heat or cold, high winds, heavy snow, ice, or flooding - begin your response with a line starting with "ALERT:" describing the hazard.

Rules:
- You only cover locations in the United States and its territories.
- If get_lat_lon returns None, tell the user you could not find that location and ask them to be more specific.
- If get_extended_weather_forecast returns None, explain that the National Weather Service does not cover that location.
- Never invent weather data. Report only what the tools return.
"""

# Logging Setup
logger = logging.getLogger("weather_agent")
logger.setLevel(logging.INFO)
try:
    handler = RotatingFileHandler("weather_agent.log", maxBytes=1_048_576, backupCount=3)
    logger.addHandler(handler)
except Exception:
    logging.basicConfig(level=logging.INFO)
    logger = logging.getLogger("weather_agent")

def log_user_prompt(callback_context: CallbackContext, llm_request: LlmRequest) -> Optional[LlmResponse]:
    if llm_request.contents:
        last = llm_request.contents[-1]
        if last.role == "user" and last.parts and last.parts[0].text:
            user_text = last.parts[0].text.strip()
            logger.info("[%s] USER >> %s", callback_context.agent_name, user_text)
            if any(w in user_text.lower() for w in ["hack", "exploit", "bomb", "ignore previous instructions"]):
                logger.warning("Blocked potential malicious prompt injection.")
                return LlmResponse(content={"role": "model", "parts": [{"text": "⚠️ Sorry, that message violates our content policies and cannot be processed."}]})
    return None

def log_model_response(callback_context: CallbackContext, llm_response: LlmResponse) -> Optional[LlmResponse]:
    if llm_response.content and llm_response.content.parts:
        txt = llm_response.content.parts[0].text
        if txt:
            logger.info("[%s] MODEL >> %s", callback_context.agent_name, txt.strip())
    return None

# Secure Weather Sub-Agent
weather_agent_with_callbacks = Agent(
    name="Pat_Secure_Agent",
    model=MODEL_GEMINI,
    description="Pat the Friendly Weather Agent with security callbacks.",
    instruction=WEATHER_AGENT_INSTRUCTIONS,
    tools=[get_lat_lon, get_extended_weather_forecast],
    before_model_callback=log_user_prompt,
    after_model_callback=log_model_response,
)

print("Weather sub-agent ready.")

Weather sub-agent ready.


In [ ]:
# @title Challenge 3: Define Search Agent
from google.adk.agents import Agent
from google.adk.tools.google_search_tool import GoogleSearchTool

# Provide the search tool natively to an isolated search agent instance
search_agent = Agent(
    name="Search_Agent",
    model=MODEL_GEMINI,
    description="Provides general web search and information lookup for non-weather queries.",
    instruction="You are a helpful search assistant. Use Google Search to find accurate, up-to-date information.",
    tools=[GoogleSearchTool()],
)

print("Search sub-agent re-initialized successfully.")

Search sub-agent re-initialized successfully.


In [ ]:
# @title Alternative: Root Coordinator with AgentTools
from google.adk.agents import Agent
from google.adk.tools import agent_tool

root_agent = Agent(
    name="Root_Coordinator",
    model=MODEL_GEMINI,
    description="Main coordinator that delegates tasks.",
    instruction="Analyze user requests and route weather queries to the weather agent and search queries to the search agent.",
    sub_agents=[weather_agent_with_callbacks],
    tools=[agent_tool.AgentTool(agent=search_agent)]
)

print("Root agent configured with mixed sub-agent and tool delegation.")

Root agent configured with mixed sub-agent and tool delegation.


In [ ]:
# @title Challenge 3: Test Execution for Multi-Agent System
from IPython.display import Markdown, display
from vertexai.preview import reasoning_engines

# 1. Initialize app with the root agent
app_multi = reasoning_engines.AdkApp(agent=root_agent)

user_id = "multi-agent-test-user"
session = app_multi.create_session(user_id=user_id)
session_id = session["id"]

# 2. Test requests spanning different capabilities
test_queries = [
    "What is the weather forecast for Miami, FL?",
    "Who won the most recent Super Bowl and what was the score?"
]

for query in test_queries:
    print(f"\nUser Query: {query}")
    print("-" * 50)

    last_event = None
    for event in app_multi.stream_query(
        user_id=user_id,
        session_id=session_id,
        message=query
    ):
        last_event = event

    if last_event and "content" in last_event and "parts" in last_event["content"]:
        response_text = last_event["content"]["parts"][0]["text"]
        display(Markdown(f"**Root Coordinator Response:**\n\n{response_text}"))
    else:
        print("No response received.")

/usr/local/lib/python3.12/dist-packages/vertexai/preview/reasoning_engines/templates/adk.py:966: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self._tmpl_attrs["credential_service"] = InMemoryCredentialService()
/usr/local/lib/python3.12/dist-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()



User Query: What is the weather forecast for Miami, FL?
--------------------------------------------------


/usr/local/lib/python3.12/dist-packages/google/adk/tools/function_tool.py:95: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  build_function_declaration(
INFO:weather_agent:[Pat_Secure_Agent] USER >> For context:
INFO:weather_agent:[Pat_Secure_Agent] MODEL >> ALERT: Heat index values are expected to reach as high as 101 degrees Fahrenheit this afternoon and up to 104 degrees Fahrenheit on Friday. There is a chance of showers and thunderstorms this afternoon and tonight, with a high near 87°F and a low around 82°F. Winds will be from the east, around 12 to 16 mph, with gusts as high as 21 mph. On Friday, expect a chance of showers and thunderstorms with a high near 88°F and a low around 83°F. The chance of showers and thunderstorms continues into Saturday, with a high near 88°F.


**Root Coordinator Response:**

ALERT: Heat index values are expected to reach as high as 101 degrees Fahrenheit this afternoon and up to 104 degrees Fahrenheit on Friday. There is a chance of showers and thunderstorms this afternoon and tonight, with a high near 87°F and a low around 82°F. Winds will be from the east, around 12 to 16 mph, with gusts as high as 21 mph. On Friday, expect a chance of showers and thunderstorms with a high near 88°F and a low around 83°F. The chance of showers and thunderstorms continues into Saturday, with a high near 88°F.

INFO:weather_agent:[Pat_Secure_Agent] USER >> Who won the most recent Super Bowl and what was the score?



User Query: Who won the most recent Super Bowl and what was the score?
--------------------------------------------------


**Root Coordinator Response:**

The most recent Super Bowl, Super Bowl LX, was won by the Seattle Seahawks, who defeated the New England Patriots with a score of 29-13. The game was played on February 8, 2026.